# 第 7 章实验：Skills 的渐进式加载与资源读取

**问题：给 Agent 配置一个 Skill 后，技能正文和参考资料会立即全部进入模型上下文吗？**

我们制作一个本地 `release-check` 技能，让 Agent 读取发布检查清单。对应[第 7 章正文](../../content/ch07-skills.md)的目录结构、渐进式披露和 `FilesystemBackend` 示例。

运行后应看到三层证据：

| 阶段 | 模型能看到什么 | 如何验证 |
|---|---|---|
| 首次请求 | 技能名、描述、`SKILL.md` 路径 | 记录实际模型请求，确认正文和资料标记尚未出现 |
| 读取技能 | `read_file` 返回的技能正文 | 对照调用参数、调用 ID、成功状态及文件内容 |
| 读取资料 | 正文指向的参考资料 | 确认资料由后续读取进入上下文 |

最后删除参考资料，用新的 Agent 重跑，观察真实的文件不存在错误。如果没有实际读取、内容提前出现，或缺失资料仍返回成功，实验会用断言报错。

本实验验证加载机制和错误反馈；真实模型是否正确选工具，需要单独运行 live 验证。

## 1. 环境与运行模式

本章可独立从第一格执行，只需要基础 Python。沿用[公共 README](../README.md) 的 Python 3.12、锁文件和模型配置，不依赖前两章的内核状态，也不需要 Agent Server。

在仓库根目录运行：

```bash
uv sync --project notebooks --locked
uv run --project notebooks --locked python -m course_notebooks.run ch07-skills
```

默认 `offline` 用脚本模型规定读取顺序，Skills 中间件、文件工具和消息循环真实执行。随附输出来自 offline，不代表真实模型的选工具能力。

要调用真实模型，按 README 配置根目录 `.env`，在命令末尾添加 `--mode live`；可能产生模型费用，配置或调用失败会直接报错。交互式运行时，先执行 `import os` 和 `os.environ["COURSE_MODE"] = "live"`，再从下一格开始顺序运行。

In [1]:
import re
from pathlib import Path
from tempfile import TemporaryDirectory

from deepagents import create_deep_agent
from deepagents.backends.filesystem import FilesystemBackend
from langchain.agents.middleware import wrap_model_call
from langchain_core.messages import AIMessage, ToolMessage

from course_notebooks.model_config import create_model
from course_notebooks.nbtools import show_runtime, show_text
from course_notebooks.testing import ScriptedChatModel

show_runtime()

运行模式： offline （脚本模型）
Python： 3.12.13 平台： Darwin arm64
deepagents==0.7.15
langchain==1.4.2
langgraph==1.2.11
langchain-openai==1.6.2


## 2. 准备一个最小技能包

`SKILL.md` 开头由 `---` 包住的部分叫 **frontmatter**，描述技能名与适用场景；后面是执行步骤。正文再指向一份参考资料。

```text
临时目录（FilesystemBackend 的 root_dir）
└── skills/
    └── release-check/
        ├── SKILL.md
        └── references/
            └── checks.md
```

`SKILL-BODY-ONLY` 是只放在正文里的观察标记；`RULE-17` 是只放在资料里的规则编号。它们帮助我们判断内容何时进入上下文。下面先定义内容，稍后在 `with TemporaryDirectory()` 内创建文件，出错时也会自动清理。

In [2]:
SKILL_PATH = "/skills/release-check/SKILL.md"
REFERENCE_PATH = "/skills/release-check/references/checks.md"
DESCRIPTION = "当用户需要发布前检查清单时，读取本地资料并列出检查项。"
BODY_MARKER = "SKILL-BODY-ONLY"
REFERENCE_MARKER = "RULE-17"

SKILL_TEXT = f"""---
name: release-check
description: {DESCRIPTION}
---
# 发布检查技能
<!-- {BODY_MARKER} -->
1. 使用 read_file 读取 `{REFERENCE_PATH}`。
2. 根据资料列出检查项及编号，不要补充资料中没有的要求。
3. 如果读取失败，报告缺失路径并停止，不要猜测检查项。
"""
REFERENCE_TEXT = """# 发布前检查清单
- RULE-17：发布前必须通过单元测试。
- RULE-23：删除构建产物中的调试日志。
"""

show_text("SKILL.md（这里只是 Python 打印，还没有传给模型）", SKILL_TEXT)
show_text("references/checks.md", REFERENCE_TEXT)


SKILL.md（这里只是 Python 打印，还没有传给模型）
---
name: release-check
description: 当用户需要发布前检查清单时，读取本地资料并列出检查项。
---
# 发布检查技能
<!-- SKILL-BODY-ONLY -->
1. 使用 read_file 读取 `/skills/release-check/references/checks.md`。
2. 根据资料列出检查项及编号，不要补充资料中没有的要求。
3. 如果读取失败，报告缺失路径并停止，不要猜测检查项。

references/checks.md
# 发布前检查清单
- RULE-17：发布前必须通过单元测试。
- RULE-23：删除构建产物中的调试日志。


## 3. 让脚本模型参与真实工具循环

`AIMessage` 是模型的回复，其中 `tool_calls` 表示模型想调用的工具；`ToolMessage` 是框架执行工具后返回的消息，`tool_call_id` 将两者关联。

离线脚本规定先读取 `SKILL.md`，再从工具返回的正文里找出资料路径；最终答复也取自工具结果。它不会直接读磁盘或构造最终状态。`create_model()` 在 live 模式会改用真实模型，这段脚本不会参与决策。

In [3]:
def scripted_reply(messages, tool_names):
    assert "read_file" in tool_names
    results = [m for m in messages if isinstance(m, ToolMessage)]
    if not results:
        path = SKILL_PATH
    elif results[-1].status == "error":
        return AIMessage(content="资料读取失败，停止处理：\n" + results[-1].text)
    elif len(results) == 1:
        # 资料路径来自实际的 SKILL.md 工具返回。
        match = re.search(r"`(/skills/[^`]+/references/[^`]+)`", results[-1].text)
        assert match, "技能正文没有给出资料路径"
        path = match.group(1)
    else:
        return AIMessage(content="读取到的检查清单：\n" + results[-1].text)
    return AIMessage(content="", tool_calls=[{
        "name": "read_file", "args": {"file_path": path},
        "id": f"read-{len(results) + 1}",
    }])

## 4. 创建 Agent，并观察真正送往模型的请求

`FilesystemBackend(root_dir=..., virtual_mode=True)` 将 `/skills/` 映射到临时目录下的 `skills/`。`skills` 传的是技能目录的**父目录**，不是 `SKILL.md` 文件。

`create_deep_agent` 会安装 Skills 中间件：扫描技能文件、解析元数据，再把技能名、描述和路径加入系统提示词。**扫描阶段已经读取文件；渐进式披露节省的是模型上下文，不是避免磁盘读取。**

`@wrap_model_call` 是观察模型请求的中间件入口。下面只记录请求，再原样交给后续处理；它位于内置 Skills 中间件之后，所以能看到注入后的提示词。记录包含系统提示词和对话消息，offline/live 使用同一观察方式。

`invoke()` 发起一次 Agent 执行：输入用户消息，框架循环调用模型与工具，返回包含 `messages` 的状态。本例限制为 12 个图执行步，避免模型持续调用工具。

In [4]:
def run_agent(root):
    requests = []

    @wrap_model_call
    def observe_request(request, handler):
        system = request.system_message.text if request.system_message else ""
        requests.append({
            "system": system,
            "context": "\n".join([system, *(m.text for m in request.messages)]),
            "tool_ids": {m.tool_call_id for m in request.messages
                         if isinstance(m, ToolMessage)},
        })
        return handler(request)

    agent = create_deep_agent(
        model=create_model(ScriptedChatModel(responder=scripted_reply)),
        backend=FilesystemBackend(root_dir=str(root), virtual_mode=True),
        skills=["/skills/"],
        middleware=[observe_request],
        system_prompt="用中文完成本地技能任务；不要联网、委派或写文件。",
    )
    result = agent.invoke({"messages": [{
        "role": "user",
        "content": "请使用 release-check 技能列出发布前检查项及编号。"
                   "先读取技能正文，再读取它指定的资料。"
                   "资料缺失时报告路径并停止，不要猜测。",
    }]}, config={"recursion_limit": 12})
    return result, requests

## 5. 执行成功场景和缺失资料场景

同一临时目录内先完成正常读取，再只删除 `checks.md`，重新创建 Agent 并发送相同请求。每次都从空对话开始，避免上一次的资料留在消息中。两个场景都结束后，临时目录自动删除；后续只检查已保留在内存里的消息。

下面打印每次 `read_file` 的路径、返回状态及内容，并保留换行。模型最终答复使用纯文本输出，里面的 Markdown 标题不会变成 Notebook 的章节标题。

In [5]:
def show_reads(label, result):
    print(f"\n=== {label} ===")
    calls = {c["id"]: c for m in result["messages"]
             if isinstance(m, AIMessage) for c in m.tool_calls}
    for message in result["messages"]:
        if isinstance(message, ToolMessage):
            call = calls[message.tool_call_id]
            show_text(call["name"], call["args"])
            print("返回状态：", message.status)
            show_text("工具返回：", message.text)
    show_text("最终答复：", result["messages"][-1].text)


with TemporaryDirectory(prefix="ch07-skills-") as directory:
    root = Path(directory)
    skill_file = root / SKILL_PATH.lstrip("/")
    reference_file = root / REFERENCE_PATH.lstrip("/")
    reference_file.parent.mkdir(parents=True)
    skill_file.write_text(SKILL_TEXT, encoding="utf-8")
    reference_file.write_text(REFERENCE_TEXT, encoding="utf-8")

    success, success_requests = run_agent(root)
    show_reads("资料存在", success)

    reference_file.unlink()
    missing, missing_requests = run_agent(root)
    show_reads("资料缺失", missing)

assert not root.exists(), "临时目录未清理"
print("\n临时技能目录已清理。")


=== 资料存在 ===

read_file
{'file_path': '/skills/release-check/SKILL.md'}
返回状态： success

工具返回：
@@ lines 1-9 of 9 @@
---
name: release-check
description: 当用户需要发布前检查清单时，读取本地资料并列出检查项。
---
# 发布检查技能
<!-- SKILL-BODY-ONLY -->
1. 使用 read_file 读取 `/skills/release-check/references/checks.md`。
2. 根据资料列出检查项及编号，不要补充资料中没有的要求。
3. 如果读取失败，报告缺失路径并停止，不要猜测检查项。

read_file
{'file_path': '/skills/release-check/references/checks.md'}
返回状态： success

工具返回：
@@ lines 1-3 of 3 @@
# 发布前检查清单
- RULE-17：发布前必须通过单元测试。
- RULE-23：删除构建产物中的调试日志。

最终答复：
读取到的检查清单：
@@ lines 1-3 of 3 @@
# 发布前检查清单
- RULE-17：发布前必须通过单元测试。
- RULE-23：删除构建产物中的调试日志。

=== 资料缺失 ===

read_file
{'file_path': '/skills/release-check/SKILL.md'}
返回状态： success

工具返回：
@@ lines 1-9 of 9 @@
---
name: release-check
description: 当用户需要发布前检查清单时，读取本地资料并列出检查项。
---
# 发布检查技能
<!-- SKILL-BODY-ONLY -->
1. 使用 read_file 读取 `/skills/release-check/references/checks.md`。
2. 根据资料列出检查项及编号，不要补充资料中没有的要求。
3. 如果读取失败，报告缺失路径并停止，不要猜测检查项。

read_file
{'file_path': '/skills/release-check/ref

## 6. 验证读取顺序和内容来源

先把 `read_file` 调用与其返回按 ID 配对。成功读取必须返回相应文件里的内容；缺失资料必须返回 `error` 状态和缺失路径。

然后检查每次模型请求：正文标记出现之前，必须已有技能读取的返回；资料标记出现之前，必须已有参考资料读取的返回。只检查最终答复，无法证明这两个边界。

In [6]:
def read_pairs(result):
    calls = {}
    pairs = []
    for message in result["messages"]:
        if isinstance(message, AIMessage):
            for call in message.tool_calls:
                assert call["id"] not in calls, "未返回的工具调用 ID 重复"
                calls[call["id"]] = call
        elif isinstance(message, ToolMessage):
            assert message.tool_call_id in calls, "工具返回没有对应调用"
            call = calls.pop(message.tool_call_id)
            assert message.name == call["name"], "工具名称不匹配"
            if call["name"] == "read_file":
                pairs.append((call, message))
    assert not calls, "工具调用缺少返回"
    return pairs

下面的检查函数用于两个场景。成功时核对资料内容和最终答复中的规则编号；失败时核对错误状态和缺失路径，检查答复没有复用本例中的规则编号。

In [7]:
def check_disclosure(result, requests, *, resource_exists):
    pairs = read_pairs(result)
    paths = [c["args"]["file_path"] for c, _ in pairs]
    assert SKILL_PATH in paths and REFERENCE_PATH in paths, "缺少必要读取"
    skill_index, ref_index = paths.index(SKILL_PATH), paths.index(REFERENCE_PATH)
    assert skill_index < ref_index, "应先读取技能正文，再读取资料"
    skill_call, skill_reply = pairs[skill_index]
    ref_call, ref_reply = pairs[ref_index]
    assert skill_reply.status == "success"
    assert all(line in skill_reply.text for line in SKILL_TEXT.splitlines())

    first = requests[0]
    assert all(value in first["system"]
               for value in ("release-check", DESCRIPTION, SKILL_PATH))
    assert BODY_MARKER not in first["context"]
    assert REFERENCE_MARKER not in first["context"]
    for request in requests:
        if skill_call["id"] not in request["tool_ids"]:
            assert BODY_MARKER not in request["context"], "技能正文提前进入上下文"
        if ref_call["id"] not in request["tool_ids"]:
            assert REFERENCE_MARKER not in request["context"], "资料提前进入上下文"
    after_skill = next(r for r in requests
                       if skill_call["id"] in r["tool_ids"])
    assert BODY_MARKER in after_skill["context"]
    assert ref_call["id"] not in after_skill["tool_ids"], "两个读取阶段未分开"
    after_ref = next(r for r in requests if ref_call["id"] in r["tool_ids"])
    final = result["messages"][-1]
    assert isinstance(final, AIMessage) and not final.tool_calls
    if resource_exists:
        assert ref_reply.status == "success"
        assert all(line in ref_reply.text for line in REFERENCE_TEXT.splitlines())
        assert REFERENCE_MARKER in after_ref["context"]
        assert all(rule in final.text for rule in ("RULE-17", "RULE-23"))
    else:
        assert ref_reply.status == "error"
        assert REFERENCE_PATH in final.text
        assert all(rule not in final.text for rule in ("RULE-17", "RULE-23"))
        assert REFERENCE_PATH in ref_reply.text and "not found" in ref_reply.text
        assert all(REFERENCE_MARKER not in r["context"] for r in requests)

In [8]:
for label, result, requests, exists in [
    ("资料存在", success, success_requests, True),
    ("资料缺失", missing, missing_requests, False),
]:
    check_disclosure(result, requests, resource_exists=exists)
    print(f"\n{label}：调用关联、文件内容和披露边界均通过")
    for index, request in enumerate(requests, 1):
        body_seen = BODY_MARKER in request["context"]
        reference_seen = REFERENCE_MARKER in request["context"]
        print(f"  请求 {index}：技能正文={body_seen}，参考规则={reference_seen}")
    # 只打印和本实验有关的提示词行，避免输出整份系统提示词。
    if exists:
        relevant = [line for line in requests[0]["system"].splitlines()
                    if "release-check" in line]
        show_text("首次请求中的技能目录片段：", "\n".join(relevant))


资料存在：调用关联、文件内容和披露边界均通过
  请求 1：技能正文=False，参考规则=False
  请求 2：技能正文=True，参考规则=False
  请求 3：技能正文=True，参考规则=True

首次请求中的技能目录片段：
- **release-check**: 当用户需要发布前检查清单时，读取本地资料并列出检查项。
  -> Read `/skills/release-check/SKILL.md` for full instructions

资料缺失：调用关联、文件内容和披露边界均通过
  请求 1：技能正文=False，参考规则=False
  请求 2：技能正文=True，参考规则=False
  请求 3：技能正文=True，参考规则=False


## 7. 改一个变量，再观察

在第 2 节的 `REFERENCE_TEXT` 中，把 `RULE-23` 的检查项改成“发布前更新变更日志”，从该节重新运行后续单元格：

- 第一次模型请求仍然只有技能目录信息，不包含修改后的资料正文。
- 成功场景的第二次 `read_file` 返回新内容；内容断言应继续通过。
- 缺失场景仍然报文件不存在，不会沿用成功场景的资料。

本实验不要求真实模型复述完全相同的措辞。需要确认它是否按要求总结时，请结合工具轨迹阅读最终答复；离线脚本只演示机械读取，不评估发布建议质量。

## 小结与清理

- `skills=["/skills/"]` 让 Skills 中间件发现技能并注入目录信息。
- 模型通过 `read_file` 逐步取得正文和参考资料；技能包没有自动把所有资料塞入第一次模型请求。
- frontmatter 能被解析，不代表它引用的资料一定存在。缺失资料会在实际读取时返回错误。
- 文件已由 `TemporaryDirectory` 清理，即使执行中途出错也会清理；没有后台服务需要关闭。重新运行第 5 节会创建全新的目录与 Agent。

继续阅读[第 7 章正文](../../content/ch07-skills.md)中的多来源技能与子 Agent 继承；下一章是[记忆管理](../../content/ch08-long-term-memory.md)。